In [ ]:
import pandas as pd

### 1. Dataset Loading

In [ ]:
# Read tracks.csv
tracks = pd.read_csv('../data/kaggle2/tracks.csv')
tracks.head()

### 2. Data Overview

In [ ]:
tracks.info()

In [ ]:
# Checking for null values 
tracks.isnull().sum()

### 3. Data Cleaning

In [ ]:
# Handle missing names
tracks['name'] = tracks['name'].fillna("Unknown Track")

**DEALING WITH "STRINGIFIED" LISTS:**

The `artists` and `id_artists` columns were imported as stringified lists. Because Python interprets these as literal strings rather than list objects, individual elements are not directly accessible for analysis.

In [ ]:
print(f"""      --- 'artists' column ---
{tracks['artists'].tail()}

      --- 'id_artists' column ---
{tracks['id_artists'].tail()}""")

In [ ]:
# 1. Clean the string by removing list symbols
tracks['artists'] = tracks['artists'].str.replace("[", "", regex=False).str.replace("]", "", regex=False).str.replace("'", "", regex=False)
tracks['id_artists'] = tracks['id_artists'].str.replace("[", "", regex=False).str.replace("]", "", regex=False).str.replace("'", "", regex=False)

# 2. Split the string to create a list
tracks['artists'] = tracks['artists'].str.split(", ")
tracks['id_artists'] = tracks['id_artists'].str.split(", ")

# 3. Extract the first element (Main Artist)
tracks['main_artist'] = tracks['artists'].str[0]
tracks['id_main_artist'] = tracks['id_artists'].str[0]

In [ ]:
print(f"""      --- 'artists' column ---
{tracks['artists'].tail()}

      --- 'id_artists' column ---
{tracks['id_artists'].tail()}""")

**Handling Inconsistent Formats:**

- This Spotify dataset presents inconsistent date formats. In some case full dates (**YYYY-MM-DD**) are present, in others year-only entries (**YYYY**). Not knowing which other formats are present, we used this code to filter out any value shorter than the standard 10 characters.
- By extracting the unique values from this subset, we were able to identify exactly which patterns were present (such as **YYYY** or **YYYY-MM**).

In [ ]:
tracks.release_date

In [ ]:
# Create a mask for strings with lengths shorter than a full date (10 characters)
# This helps us identify years (length 4) and month-year formats (length 7)
anomaly_mask = (tracks['release_date'].str.len() < 10)

# Display unique values to understand the specific formats present
anomalous_values = tracks.loc[anomaly_mask, 'release_date'].unique()
print(anomalous_values)

In [ ]:
# --- STEP 1: Handle year-only dates (YYYY) ---
# Check for strings with 4 characters and set them to 01-01
mask_year = tracks['release_date'].str.len() == 4
tracks.loc[mask_year, 'release_date'] = tracks.loc[mask_year, 'release_date'] + '-01-01'

# --- STEP 2: Handle year-month dates (YYYY-MM) ---
# Check for strings with 7 characters and set them to the 01 day of that month
mask_month = tracks['release_date'].str.len() == 7
tracks.loc[mask_month, 'release_date'] = tracks.loc[mask_month, 'release_date'] + '-01'

# --- STEP 3: Final Conversion ---
# Now that formats are standardized to YYYY-MM-DD, convert to datetime objects
tracks['release_date'] = pd.to_datetime(tracks['release_date'], errors='coerce')

# Check if any NaT (errors) remain
print(f"Remaining invalid dates: {tracks['release_date'].isna().sum()}")

In [ ]:
# Extract year
tracks['year'] = tracks['release_date'].dt.year

In [ ]:
# Drop any duplicate record to avoid distortions
tracks = tracks.drop_duplicates(subset=['id'])

In [ ]:
print(tracks.info())
tracks.head()

### 4. Data Export

In [ ]:
tracks.to_csv("../data/tracks_clean.csv", index=False)